In [11]:
# ============================================================
# 05_recommendation.ipynb
# Sentiment-Based Hotel Recommendation System
# Uses DistilBERT predictions from 04b_inference (Kaggle)
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

print("Imports done.")

Imports done.


In [12]:
# Load the predictions produced by DistilBERT on Kaggle
# This CSV contains every review with its DistilBERT-predicted sentiment
predictions_path = '../data/processed/hotel_reviews_with_predictions.csv'

df = pd.read_csv(predictions_path)

print(f"Total reviews        : {len(df):,}")
print(f"Columns              : {list(df.columns)}")
print(f"Hotels               : {df['hotel'].nunique():,}")
print(f"\nDistilBERT prediction distribution:")
print(df['distilbert_sentiment'].value_counts())
print(f"\nScore-based label distribution:")
print(df['label'].value_counts())
print(f"\nSample rows:")
print(df.head(5).to_string(index=False))

Total reviews        : 515,287
Columns              : ['hotel', 'score', 'label', 'distilbert_sentiment']
Hotels               : 1,492

DistilBERT prediction distribution:
distilbert_sentiment
Positive    271593
Neutral     198831
Negative     44863
Name: count, dtype: int64

Score-based label distribution:
label
Positive    335338
Neutral     149281
Negative     30668
Name: count, dtype: int64

Sample rows:
                            hotel  score    label distilbert_sentiment
                    Hilton Vienna   10.0 Positive             Positive
          Melia White House Hotel    9.2 Positive             Positive
DoubleTree By Hilton London Excel   10.0 Positive             Positive
           H10 Metropolitan 4 Sup    9.2 Positive             Positive
  Eurostars Grand Marina Hotel GL   10.0 Positive              Neutral


In [13]:
# ── How often do DistilBERT and score-based labels agree? ─────
# This is the key insight: they are NOT the same system
agreement = (df['label'] == df['distilbert_sentiment']).mean() * 100
print(f"Overall agreement between rating-based and DistilBERT: {agreement:.1f}%")
print(f"Disagreement rate: {100-agreement:.1f}% of reviews")
print(f"That is {int(len(df)*(100-agreement)/100):,} reviews where the two systems differ\n")

# Cross-tabulation
ct = pd.crosstab(
    df['label'],
    df['distilbert_sentiment'],
    normalize='index'
) * 100

print("Cross-tabulation (rows=score label, cols=DistilBERT prediction, values=%):")
print(ct.round(1).to_string())
print("\nKey finding: Neutral reviews (score 6-7) are the most disputed.")
print("DistilBERT reads the text and reclassifies many as Positive or Negative.")

Overall agreement between rating-based and DistilBERT: 72.2%
Disagreement rate: 27.8% of reviews
That is 143,143 reviews where the two systems differ

Cross-tabulation (rows=score label, cols=DistilBERT prediction, values=%):
distilbert_sentiment  Negative  Neutral  Positive
label                                            
Negative                  71.8     26.1       2.1
Neutral                   12.5     70.3      17.2
Positive                   1.2     25.6      73.1

Key finding: Neutral reviews (score 6-7) are the most disputed.
DistilBERT reads the text and reclassifies many as Positive or Negative.


In [14]:
# ── Aggregate by hotel — compute stats for both systems ───────
hotel_stats = df.groupby('hotel').agg(
    n_reviews = ('distilbert_sentiment', 'count'),

    # DistilBERT-based counts
    bert_positive = ('distilbert_sentiment', lambda x: (x == 'Positive').sum()),
    bert_neutral  = ('distilbert_sentiment', lambda x: (x == 'Neutral').sum()),
    bert_negative = ('distilbert_sentiment', lambda x: (x == 'Negative').sum()),

    # Rating-based stats
    avg_score     = ('score', 'mean'),
    score_positive = ('label', lambda x: (x == 'Positive').sum()),
    score_negative = ('label', lambda x: (x == 'Negative').sum()),

).reset_index()

# DistilBERT percentages
hotel_stats['bert_pct_positive'] = hotel_stats['bert_positive'] / hotel_stats['n_reviews'] * 100
hotel_stats['bert_pct_negative'] = hotel_stats['bert_negative'] / hotel_stats['n_reviews'] * 100
hotel_stats['bert_pct_neutral']  = hotel_stats['bert_neutral']  / hotel_stats['n_reviews'] * 100

# Sentiment score = % positive - % negative (ranges -100 to +100)
hotel_stats['sentiment_score'] = (
    hotel_stats['bert_pct_positive'] - hotel_stats['bert_pct_negative']
)

# Rating-based percentages
hotel_stats['score_pct_positive'] = hotel_stats['score_positive'] / hotel_stats['n_reviews'] * 100
hotel_stats['score_pct_negative'] = hotel_stats['score_negative'] / hotel_stats['n_reviews'] * 100

# Keep only hotels with at least 10 reviews
hotel_stats = hotel_stats[hotel_stats['n_reviews'] >= 10].reset_index(drop=True)

print(f"Hotels with ≥10 reviews: {len(hotel_stats):,}")
print(f"\nSentiment score range  : {hotel_stats['sentiment_score'].min():.1f} to {hotel_stats['sentiment_score'].max():.1f}")
print(f"Average score range    : {hotel_stats['avg_score'].min():.2f} to {hotel_stats['avg_score'].max():.2f}")
print(f"\nTop 5 by sentiment score:")
print(hotel_stats.nlargest(5, 'sentiment_score')[
    ['hotel','n_reviews','bert_pct_positive','bert_pct_negative','sentiment_score','avg_score']
].to_string(index=False))

Hotels with ≥10 reviews: 1,491

Sentiment score range  : -52.3 to 92.4
Average score range    : 5.12 to 9.72

Top 5 by sentiment score:
                            hotel  n_reviews  bert_pct_positive  bert_pct_negative  sentiment_score  avg_score
            Hotel du Petit Moulin         66          92.424242           0.000000        92.424242   9.312121
                               41        103          92.233010           0.000000        92.233010   9.711650
Hollmann Beletage Design Boutique        102          92.156863           0.980392        91.176471   9.553922
       Milestone Hotel Kensington         86          91.860465           1.162791        90.697674   9.572093
                Hotel Casa Camper        301          91.029900           0.664452        90.365449   9.718937


In [15]:
def recommend_by_sentiment(top_k=5, min_reviews=50):
    """
    System 2 — DistilBERT sentiment-based recommendation.
    
    Ranks hotels by sentiment_score = % positive reviews - % negative reviews
    as predicted by DistilBERT from the actual review text.
    
    Higher score = more guests expressed positive sentiment in their writing.
    """
    candidates = hotel_stats[hotel_stats['n_reviews'] >= min_reviews].copy()
    candidates = candidates.sort_values('sentiment_score', ascending=False)

    result = candidates.head(top_k)[
        ['hotel', 'n_reviews', 'bert_pct_positive',
         'bert_pct_negative', 'sentiment_score', 'avg_score']
    ].reset_index(drop=True)

    result.columns = ['Hotel', 'Reviews', '% Positive (BERT)',
                      '% Negative (BERT)', 'Sentiment Score', 'Avg Rating']
    result.index  += 1
    return result

print("="*65)
print("SYSTEM 2 — TOP 10 HOTELS (DistilBERT Sentiment-Based)")
print("="*65)
print(recommend_by_sentiment(top_k=10, min_reviews=50).to_string())

SYSTEM 2 — TOP 10 HOTELS (DistilBERT Sentiment-Based)
                                    Hotel  Reviews  % Positive (BERT)  % Negative (BERT)  Sentiment Score  Avg Rating
1                   Hotel du Petit Moulin       66          92.424242           0.000000        92.424242    9.312121
2                                      41      103          92.233010           0.000000        92.233010    9.711650
3       Hollmann Beletage Design Boutique      102          92.156863           0.980392        91.176471    9.553922
4              Milestone Hotel Kensington       86          91.860465           1.162791        90.697674    9.572093
5                       Hotel Casa Camper      301          91.029900           0.664452        90.365449    9.718937
6                            H tel Fabric      103          90.291262           0.000000        90.291262    9.559223
7   Pillows Anna van den Vondel Amsterdam       56          91.071429           1.785714        89.285714    9.366071
8 

In [16]:
# Export for Khaoula's comparison notebook (07_comparison.ipynb)
hotel_scores_path = '../results/hotel_scores.csv'

export_cols = [
    'hotel', 'n_reviews',
    'bert_pct_positive', 'bert_pct_negative', 'bert_pct_neutral',
    'sentiment_score',
    'avg_score',
    'score_pct_positive', 'score_pct_negative'
]

hotel_stats.sort_values('sentiment_score', ascending=False)[export_cols].to_csv(
    hotel_scores_path, index=False
)

print(f"Exported: {hotel_scores_path}")
print(f"Shape   : {hotel_stats.shape}")
print(f"\nTop 5 hotels by sentiment score:")
print(hotel_stats.nlargest(5, 'sentiment_score')[
    ['hotel','n_reviews','sentiment_score','avg_score']
].to_string(index=False))
print(f"\nBottom 5 hotels by sentiment score:")
print(hotel_stats.nsmallest(5, 'sentiment_score')[
    ['hotel','n_reviews','sentiment_score','avg_score']
].to_string(index=False))

Exported: ../results/hotel_scores.csv
Shape   : (1491, 14)

Top 5 hotels by sentiment score:
                            hotel  n_reviews  sentiment_score  avg_score
            Hotel du Petit Moulin         66        92.424242   9.312121
                               41        103        92.233010   9.711650
Hollmann Beletage Design Boutique        102        91.176471   9.553922
       Milestone Hotel Kensington         86        90.697674   9.572093
                Hotel Casa Camper        301        90.365449   9.718937

Bottom 5 hotels by sentiment score:
                                   hotel  n_reviews  sentiment_score  avg_score
                           Hotel Liberty         65       -52.307692   5.121538
                           Villa Eugenie         62       -25.806452   5.864516
                   Savoy Hotel Amsterdam        243       -23.868313   6.009465
                 Villa Lut ce Port Royal         47       -23.404255   6.385106
Best Western Maitrise Hotel Edgw